# Claude Code: Baseline vs ALDC — BCApps-5633

Comparación de 3 configuraciones de Claude Code sobre el mismo problema (Shopify third-party fulfillment location).

- **baseline**: Claude Code + altool MCP, sin custom instructions ni skills
- **aldc-al-developer-bench**: Claude Code + ALDC (custom instructions + skills + al-developer-bench agent)
- **aldc-sonnet-4-6**: Claude Code + ALDC con configuración alternativa (populating al-conductor dir)

**Nota**: Un único instance — análisis cualitativo, sin significancia estadística.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RESULT_DIR = Path.cwd().parent / "result" / "bug-fix"

SETUPS = {
    "baseline": RESULT_DIR / "baseline-sonnet-4-6",
    "aldc-al-developer-bench": RESULT_DIR / "aldc-al-developer-bench-sonnet-4-6",
    "aldc-sonnet-4-6": RESULT_DIR / "aldc-sonnet-4-6",
}

LABELS = {
    "baseline": "Baseline (no ALDC)",
    "aldc-al-developer-bench": "ALDC + al-developer-bench",
    "aldc-sonnet-4-6": "ALDC (conductor dir)",
}


def load_result(folder: Path) -> dict:
    for f in folder.glob("*.jsonl"):
        line = f.read_text(encoding="utf-8").strip()
        return json.loads(line)
    raise FileNotFoundError(f"No jsonl in {folder}")


results = {k: load_result(v) for k, v in SETUPS.items()}

for k, r in results.items():
    print(f"{LABELS[k]:35s} | resolved={r['resolved']} | turns={r['metrics']['turn_count']} | tokens={r['metrics']['prompt_tokens']:,}")

## 1. Comparación de métricas de eficiencia

In [ ]:
rows = []
for k, r in results.items():
    m = r["metrics"]
    exp = r["experiment"]
    rows.append({
        "Setup": LABELS[k],
        "Resolved": "✅" if r["resolved"] else "❌",
        "Build": "✅" if r["build"] else "❌",
        "Tiempo (s)": round(m["execution_time"]),
        "Turns": m["turn_count"],
        "Prompt tokens": m["prompt_tokens"],
        "Completion tokens": m["completion_tokens"],
        "Custom instructions": exp["custom_instructions"],
        "Skills": exp["skills_enabled"],
        "Agent": exp["custom_agent"] or "—",
    })

pd.DataFrame(rows)

In [ ]:
baseline_tokens = results["baseline"]["metrics"]["prompt_tokens"]
colors = ["#2ecc71" if r["resolved"] else "#e74c3c" for r in results.values()]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[LABELS[k] for k in results],
    y=[r["metrics"]["prompt_tokens"] / 1000 for r in results.values()],
    marker_color=colors,
    text=[f"{'✅' if r['resolved'] else '❌'} {r['metrics']['prompt_tokens']/1000:.0f}K" for r in results.values()],
    textposition="outside",
))

fig.update_layout(
    title="Prompt Tokens por configuración (verde=resolved, rojo=failed)",
    yaxis_title="Prompt tokens (K)",
    xaxis_title="",
    height=400,
    showlegend=False,
)
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Turns", "Tiempo de ejecución (s)"))

names = [LABELS[k] for k in results]
turns = [r["metrics"]["turn_count"] for r in results.values()]
times = [r["metrics"]["execution_time"] for r in results.values()]
colors = ["#2ecc71" if r["resolved"] else "#e74c3c" for r in results.values()]

fig.add_trace(go.Bar(x=names, y=turns, marker_color=colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=names, y=times, marker_color=colors, showlegend=False), row=1, col=2)

fig.update_layout(height=350, title_text="Eficiencia: menos = mejor (verde=resolved)")
fig.show()

## 2. Análisis del parche generado

El problema (BCApps-5633): al exportar envíos de Shopify, se enviaban fulfillments a
localizaciones de terceros que no deberían procesarse internamente.

In [ ]:
from unidiff import PatchSet


def analyze_patch(patch_str: str) -> dict:
    try:
        ps = PatchSet(patch_str)
        return {
            "files_changed": len(ps),
            "lines_added": sum(f.added for f in ps),
            "lines_removed": sum(f.removed for f in ps),
            "files": [f.path for f in ps],
        }
    except Exception as e:
        return {"error": str(e)}


for k, r in results.items():
    info = analyze_patch(r.get("generated_patch", ""))
    print(f"\n{'='*60}")
    print(f"{LABELS[k]} — Resolved: {r['resolved']}")
    print(f"  Files changed : {info.get('files_changed', '?')}")
    print(f"  Lines added   : {info.get('lines_added', '?')} | removed: {info.get('lines_removed', '?')}")
    for f in info.get("files", []):
        print(f"    - {Path(f).name}")

## 3. ¿Por qué falló ALDC?

### Baseline (RESOLVED ✅)

Fix en el lugar **exacto**: dentro del bucle que acumula `FulfillmentOrderLine` en
`CreateFulfillmentOrderRequest`. Usa datos ya existentes en `ShopLocation."Is Fulfillment Service"`.

```al
if IsThirdPartyFulfillmentLocation(Shop, FulfillmentOrderLine."Shopify Location Id") then
    continue;
```

Solo modifica **1 archivo**, **+9 líneas**.

---

### ALDC al-developer-bench (FAILED ❌)

Check aplicado demasiado arriba en `CreateShopifyFulfillment()` usando el `LocationId` del
envío (no el del `FulfillmentOrderLine`). La prueba sigue viendo `FulfillmentRequest count = 1`
porque el bloqueo ocurre antes de donde el test espera que no llegue.

---

### ALDC sonnet-4-6 / conductor (FAILED ❌)

Enfoque sobre-ingenierizado:
- Modifica la GraphQL query para añadir `supportedActions`
- Añade campo `Third Party` a la tabla `FulfillmentOrderHeader`
- Rellena ese campo desde la API de Shopify en tiempo de ejecución

El problema: los tests de BC usan **mocks** que no llaman a la API real, por lo que
`Third Party = false` siempre y el filtro nunca actúa. **58 turns y 3M tokens** para
una solución que no puede pasar los tests unitarios.

In [ ]:
patch_analysis = {k: analyze_patch(r.get("generated_patch", "")) for k, r in results.items()}

summary = []
for k, r in results.items():
    pa = patch_analysis[k]
    overhead = f"+{(r['metrics']['prompt_tokens'] - baseline_tokens)/1000:.0f}K" if k != "baseline" else "—"
    summary.append({
        "Configuración": LABELS[k],
        "Resultado": "RESOLVED ✅" if r["resolved"] else "FAILED ❌",
        "Archivos modificados": pa.get("files_changed", "?"),
        "Líneas añadidas": pa.get("lines_added", "?"),
        "Prompt tokens": f"{r['metrics']['prompt_tokens']:,}",
        "Turns": r["metrics"]["turn_count"],
        "Tiempo (s)": round(r["metrics"]["execution_time"]),
        "Overhead vs baseline": overhead,
    })

pd.DataFrame(summary).set_index("Configuración")

## 4. Conclusiones

| Hallazgo | Detalle |
|----------|---------|
| **ALDC añade coste sin beneficio** | +70% a +330% tokens vs baseline, sin resolver el problema |
| **Custom instructions + skills interfieren** | El agente ALDC generó soluciones más complejas y perdió el foco del lugar exacto del fix |
| **Simplicidad > completitud** | Fix correcto = 9 líneas en 1 archivo. ALDC conductor = 50+ líneas en 5 archivos |
| **Dependencia de API en tests** | Soluciones que dependen de datos de API fallan contra mocks de BC |
| **Muestra = 1 instance** | Resultados indicativos; necesitamos más instancias para conclusiones definitivas |

### Hipótesis para investigar con más instancias
- ¿Las custom instructions de ALDC empujan al agente hacia soluciones más complejas que son incorrectas para tests unitarios?
- ¿El skill de AL hace que el agente sobre-explore el código de BC antes de hacer cambios?
- ¿Baseline con solo altool MCP es el punto de partida óptimo para bug-fix?